# **NYT Crosswords - Skill Evolution**

I have been a diligent solver of the NYT daily crosswords for over 10 years now. 

I was introduced to American style crosswords during my Michigan days with the daily crossword in the Michigan Daily. I subscribed to New York Times and started solving their crosswords in 2016. I was able to solve the early-week puzzles (Mon-Wed) reasonably well but struggled with late-week puzzles. In an attempt to get better at the Fri/Sat themeless puzzles, I started solving some puzzles from the archives.

At some point, this turned into a methodical process where, in addition to solving the day's xword, I would also solve a couple from the archive. And, one fateful day, the thought came unbidden into my head that I should solve _all_ the puzzles available in the archive before my 40th birthday (which was then a couple of years away). And the rest, as they say, is history!!!

Between these archival solves and the daily ones, I now have a dataset of nearly 12,000 solved puzzles. This project is an attempt to track how my crossword solving time has evolved over the years and learn some regression modeling in the process.

Note: Outputs from some of the cells below have been removed because they contain widgets that don't play nicely with Github hosting

<a id="top"></a>

**Contents**

1. [The Modeling Task](#modeling-task)
2. [Data at a Glance](#data)
3. [A Simple OLS Model](#ols)
4. [Mixed-effect Modeling](#mixed)
5. [Bayesian Models](#bayesian)
6. [Model Interpretation](#interpretation)
7. [Compare Models](#compare)
8. [Prediction Breakdown](#breakdown)
9. [Big Takeaways](#takeaways)
10. [What Next](#next)


<a id="modeling-task"></a>

## The Modeling Task

The time it takes for me to solve any given crossword depends on a bunch of different factors.


*   My innate **crossword solving ability** (which is what we are trying to model)
*   **Day of the week**. Monday crosswords are easiest and the difficulty steadily increases through the week with Saturdays being toughest. Sundays are comparable to Thursdays in difficulty (a cool outcome of this exercise is presenting mathematical evidence of this fact) but feature a larger grid
*   **Puzzle fustiness**. Solving a puzzle from 1995 in 2021 introduces some "fustiness". The grid might feature answers that were popular back then but are no longer in vogue. Also, older crosswords were objectively harder (obscure proper nouns, difficult crossings, lots of crosswordese, and tougher cluing overall). For instance, the kind of entries that show up disproportionately in older grids (ESNE, SMEE, ETUI, ADIT, ONAGER ...) has fallen out of favor as editors modernized the fill. Add to that dated proper nouns (bygone actors, defunct brands, Cold-War-era geography such as USSR/SSR/ULAN), references to news cycles long past, and cluing conventions that leaned more on rote trivia and less on the wordplay and misdirection that define modern puzzles. A 1995 grid solved in 2021 is hard on two counts: its vocabulary is stale *and* its cluing plays harder.
*   **Puzzle constructor**. Some constructors are known for their brutal grids while others play easier. But this is a difficult signal to learn because the vast majority of constructors would have been published only a handful of times (often just once)
*   **Puzzle attributes**. How difficult is the fill and the cluing? Are there difficult words, tricky clues, misdirection? This is nearly impossible to quantify and I have some limited proxies for this.
*   **Everything else**!! The signal here is inherently noisy with solve times being impacted by any number of other variables. Maybe I had a headache while solving one day, maybe I knew an obscure, long answer down cold which opened up the entire grid.


We have relatively few parameters and lots of data to estimate them with. Lets try to build interpretable models that can isolate the evolution of my crossword solving ability while controlling for each of these confounding factors.

In [35]:
#@title Install necessary packages
#%pip install -r requirements.txt

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.lines import Line2D
import itertools
from sklearn.preprocessing import LabelEncoder
import statsmodels.api as sm
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor
import seaborn as sns

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.frameon": False,
})
%matplotlib inline

# ----------------------------------------------------------------------
# Shared constants used across the OLS, mixed-effects and Bayesian sections.
# Defining them once here keeps colours and labels consistent everywhere and
# avoids the copy-paste redundancy of the original per-section notebooks.
# ----------------------------------------------------------------------
PALETTE = {
    "ols":     "#4c72b0",   # blue   - OLS
    "mixed":   "#dd8452",   # orange - mixed effects
    "bayes":   "#55a868",   # green  - Bayesian
    "primary": "#008b8b",   # darkcyan - single-model highlight
    "accent":  "#c44e52",   # crimson
    "muted":   "#708090",   # slategray
    "start":   "#009E73",   # journey-start marker
    "archive": "#D55E00",   # archive-sweep markers
    "pos":     "#c44e52",   # slower than average
    "neg":     "#55a868",   # faster than average
}
day_labels_full = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                   'Friday', 'Saturday', 'Sunday']

<a id="data"></a>
[&#8593; Back to top](#top)

## Data at a Glance

Load all the puzzle data. There is a small fraction of puzzles that are missing some puzzle features. Remove them and sort the remaining rows by order of date of solving.

In [36]:
#@title Read xword solve times data
df = pd.read_csv("solve_data/DataForModeling_Upto2025.csv")

df['PuzzleDate'] = pd.to_datetime(df['PuzzleDate'], format="%d-%m-%Y", errors='coerce')
df['SolveDate']  = pd.to_datetime(df['SolveDate'],  format="%d-%m-%Y", errors='coerce')

continuous_features = ['AvgWordLen', 'P90ScrabbleScore']
required = ['PuzzleDate', 'SolveDate', 'SolveTime', 'PuzzleDay', 'Constructor'] + continuous_features

before_rows = len(df)
df = df.dropna(subset=required).reset_index(drop=True)
after_rows = len(df)

print(f"Historic data prepared. Dropped {before_rows - after_rows} rows with missing fields.")
print(f"Total historic rows: {after_rows}")

df = df[['PuzzleDate','SolveDate','PuzzleDay','SolveTime','AvgWordLen','P90ScrabbleScore','Constructor']]

Historic data prepared. Dropped 0 rows with missing fields.
Total historic rows: 11661


In [37]:
df.head()

,PuzzleDate,SolveDate,PuzzleDay,SolveTime,AvgWordLen,P90ScrabbleScore,Constructor
0,2016-06-01,2016-06-01,Wednesday,466,4.769231,2.075,Wren Schultz
1,2016-06-02,2016-06-02,Thursday,1062,5.153846,3.000,Susan Gelfand
2,2013-05-16,2016-06-02,Thursday,764,5.175676,2.650,Brendan Emmett Quigley and Elizabeth Donovan
3,2016-06-06,2016-06-06,Monday,328,4.846154,2.130,Mary Lou Guizzo
4,2016-06-05,2016-06-06,Sunday,952,5.152174,2.200,Tom McCoy


### Solve history

This is how my solving journey looks